In [3]:
# ============================================
# MARKET INTELLIGENCE PRO - VERSÃO UNIFICADA
# Sistema completo de web scraping, análise de mercado com IA e vetorização semântica
# ============================================

print("🚀 Instalando dependências do Market Intelligence Pro Unificado...")

# Instalar todas as bibliotecas necessárias
get_ipython().system('pip install flask flask-cors playwright nest_asyncio google-generativeai PyPDF2 openpyxl sentence-transformers scikit-learn pandas numpy matplotlib seaborn -q')

# Instalar Playwright e navegador Chromium
get_ipython().system('playwright install chromium')
get_ipython().system('playwright install-deps')

print("✅ Instalação concluída!")
print("\n" + "="*60)
print("📊 MARKET INTELLIGENCE PRO - UNIFIED")
print("Sistema de Web Scraping, Análise com IA e Vetorização Semântica")
print("="*60 + "\n")

# ============================================
# IMPORTS
# ============================================

import threading
import nest_asyncio
import json
import os
from flask import Flask, jsonify, request
from flask_cors import CORS
from playwright.sync_api import sync_playwright
import google.generativeai as genai
from werkzeug.utils import secure_filename
import PyPDF2
import openpyxl
from io import BytesIO

# Bibliotecas para análise e vetorização
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
from sentence_transformers import SentenceTransformer
import warnings

warnings.filterwarnings("ignore")
nest_asyncio.apply()

# ============================================
# CONFIGURAÇÃO DO FLASK
# ============================================

app = Flask(__name__)
CORS(app)

# Configurações
UPLOAD_FOLDER = '/tmp/uploads'
DATA_FOLDER = '/tmp/data'
ALLOWED_EXTENSIONS = {'txt', 'pdf', 'xlsx', 'xls', 'csv'}

os.makedirs(UPLOAD_FOLDER, exist_ok=True)
os.makedirs(DATA_FOLDER, exist_ok=True)

app.config['UPLOAD_FOLDER'] = UPLOAD_FOLDER
app.config['DATA_FOLDER'] = DATA_FOLDER
app.config['MAX_CONTENT_LENGTH'] = 16 * 1024 * 1024  # 16MB max

# ============================================
# INICIALIZAÇÃO DO MODELO DE EMBEDDINGS
# ============================================

print("🤖 Carregando modelo de embeddings semânticos...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Modelo de embeddings carregado!")

# Variável global para armazenar dados processados
marketing_data = None

# ============================================
# FUNÇÕES AUXILIARES
# ============================================

def allowed_file(filename):
    """Verifica se o arquivo tem extensão permitida"""
    return '.' in filename and filename.rsplit('.', 1)[1].lower() in ALLOWED_EXTENSIONS


def extract_text_from_pdf(file_stream):
    """Extrai texto de arquivo PDF"""
    try:
        pdf_reader = PyPDF2.PdfReader(file_stream)
        text = ""
        for page in pdf_reader.pages:
            text += page.extract_text() + "\n"
        return text
    except Exception as e:
        return f"Erro ao extrair PDF: {str(e)}"


def extract_text_from_excel(file_stream):
    """Extrai texto de arquivo Excel"""
    try:
        workbook = openpyxl.load_workbook(file_stream)
        text = ""
        for sheet_name in workbook.sheetnames:
            sheet = workbook[sheet_name]
            text += f"\n=== {sheet_name} ===\n"
            for row in sheet.iter_rows(values_only=True):
                row_text = '\t'.join([str(cell) if cell is not None else '' for cell in row])
                text += row_text + "\n"
        return text
    except Exception as e:
        return f"Erro ao extrair Excel: {str(e)}"


def process_marketing_data(df):
    """
    Processa dataset de marketing com vetorização semântica
    Retorna dataset enriquecido e análises
    """
    global marketing_data

    print("\n📊 Iniciando processamento de dados de marketing...")

    # Limpeza inicial
    df = df.drop_duplicates()

    # Identificar coluna de texto (ad_copy, campaign_name, etc)
    text_columns = [col for col in df.columns if df[col].dtype == 'object']

    if not text_columns:
        raise ValueError("Nenhuma coluna de texto encontrada no dataset")

    # Usar primeira coluna de texto como principal
    text_column = text_columns[0]
    df[text_column] = df[text_column].fillna("").astype(str)

    print(f"✅ Coluna de texto identificada: {text_column}")

    # Identificar colunas numéricas para métricas
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    # Tratar valores nulos em colunas numéricas
    for col in numeric_cols:
        df[col] = df[col].fillna(0)

    # Gerar embeddings vetoriais
    print("🔄 Gerando embeddings semânticos...")
    embeddings = embedding_model.encode(df[text_column].tolist())
    df["embedding"] = embeddings.tolist()

    # Criar features derivadas se houver colunas de marketing típicas
    if 'impressions' in df.columns and 'clicks' in df.columns:
        df["ctr"] = np.where(df["impressions"] > 0,
                            df["clicks"] / df["impressions"], 0)
        print("✅ CTR calculado")

    if 'spend' in df.columns and 'conversions' in df.columns:
        df["cpa"] = np.where(df["conversions"] > 0,
                            df["spend"] / df["conversions"], 0)
        print("✅ CPA calculado")

    if 'spend' in df.columns and 'clicks' in df.columns:
        df["cpc"] = np.where(df["clicks"] > 0,
                            df["spend"] / df["clicks"], 0)
        print("✅ CPC calculado")

    # Clusterização semântica
    print("🔄 Aplicando clusterização...")
    n_clusters = min(5, len(df))  # Ajustar número de clusters baseado no tamanho
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    df["text_cluster"] = kmeans.fit_predict(embeddings)

    # Matriz de similaridade
    similarity_matrix = cosine_similarity(embeddings)

    # Análise por cluster
    cluster_analysis = df.groupby("text_cluster").agg({
        col: 'mean' for col in numeric_cols if col in df.columns
    }).reset_index()

    # Redução dimensional para visualização
    pca = PCA(n_components=2)
    reduced = pca.fit_transform(embeddings)
    df["pca_x"] = reduced[:, 0]
    df["pca_y"] = reduced[:, 1]

    # Armazenar dados processados globalmente
    marketing_data = {
        'dataframe': df,
        'embeddings': embeddings,
        'similarity_matrix': similarity_matrix,
        'text_column': text_column,
        'cluster_analysis': cluster_analysis
    }

    print("✅ Processamento concluído!")

    return df, cluster_analysis.to_dict('records')


def semantic_search(query, top_n=5):
    """Busca semântica no dataset processado"""
    global marketing_data

    if marketing_data is None:
        return None

    df = marketing_data['dataframe']
    embeddings = marketing_data['embeddings']
    text_column = marketing_data['text_column']

    # Gerar embedding da query
    query_vec = embedding_model.encode([query])

    # Calcular similaridades
    sims = cosine_similarity(query_vec, embeddings)[0]

    # Top N mais similares
    top_indices = sims.argsort()[-top_n:][::-1]

    results = df.iloc[top_indices].copy()
    results['similarity_score'] = sims[top_indices]

    return results


def find_similar_items(index, top_n=5):
    """Encontra itens similares a um índice específico"""
    global marketing_data

    if marketing_data is None:
        return None

    similarity_matrix = marketing_data['similarity_matrix']
    df = marketing_data['dataframe']

    sim_scores = list(enumerate(similarity_matrix[index]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    similar_indices = [i[0] for i in sim_scores[1:top_n+1]]

    return df.iloc[similar_indices]


# ============================================
# ENDPOINTS DA API
# ============================================

@app.route('/')
def home():
    """Rota principal - Status do servidor"""
    return jsonify({
        'status': 'online',
        'service': 'Market Intelligence Pro - Unified',
        'version': '2.0',
        'features': [
            'Web Scraping',
            'File Upload & Processing',
            'AI Analysis (Gemini)',
            'Semantic Vectorization',
            'Clustering & Similarity Search',
            'Marketing Data Analysis'
        ],
        'endpoints': {
            'scrape': '/scrape [POST]',
            'analyze': '/analyze [POST]',
            'upload': '/upload [POST]',
            'upload_marketing_data': '/upload_marketing_data [POST]',
            'process_marketing': '/process_marketing [POST]',
            'semantic_search': '/semantic_search [POST]',
            'find_similar': '/find_similar [POST]',
            'get_clusters': '/get_clusters [GET]',
            'get_analysis': '/get_analysis [GET]'
        }
    })


@app.route('/scrape', methods=['POST'])
def scrape():
    """Endpoint de web scraping"""
    try:
        data = request.json
        url = data.get('url')

        if not url:
            return jsonify({'error': 'URL não fornecida'}), 400

        if not url.startswith(('http://', 'https://')):
            return jsonify({'error': 'URL inválida. Use http:// ou https://'}), 400

        print(f"🌐 Fazendo scraping de: {url}")

        with sync_playwright() as p:
            browser = p.chromium.launch(headless=True)
            page = browser.new_page()

            page.set_default_timeout(30000)
            page.goto(url, wait_until='networkidle')

            title = page.title()
            paragraphs = page.locator('p').all_inner_texts()
            content = '\n\n'.join(paragraphs)

            h1_texts = page.locator('h1').all_inner_texts()
            h2_texts = page.locator('h2').all_inner_texts()
            meta_description = page.locator('meta[name="description"]').get_attribute('content') or ''

            links = []
            try:
                link_elements = page.locator('a').all()
                for link in link_elements[:20]:
                    href = link.get_attribute('href')
                    text = link.inner_text()
                    if href and text:
                        links.append({'text': text.strip(), 'href': href})
            except:
                pass

            browser.close()

            print(f"✅ Scraping concluído: {len(content)} caracteres extraídos")

            result = {
                'success': True,
                'url': url,
                'title': title,
                'content': content,
                'metadata': {
                    'meta_description': meta_description,
                    'h1_headings': h1_texts,
                    'h2_headings': h2_texts,
                    'links': links,
                    'content_length': len(content)
                }
            }

            return jsonify(result)

    except Exception as e:
        print(f"❌ Erro no scraping: {str(e)}")
        return jsonify({
            'success': False,
            'error': str(e),
            'message': 'Erro ao fazer scraping da página'
        }), 500


@app.route('/upload', methods=['POST'])
def upload_file():
    """Endpoint de upload de arquivos gerais"""
    try:
        if 'file' not in request.files:
            return jsonify({'error': 'Nenhum arquivo enviado'}), 400

        file = request.files['file']

        if file.filename == '':
            return jsonify({'error': 'Nome de arquivo vazio'}), 400

        if file and allowed_file(file.filename):
            filename = secure_filename(file.filename)
            file_ext = filename.rsplit('.', 1)[1].lower()

            print(f"📁 Processando arquivo: {filename}")

            text_content = ""

            if file_ext == 'pdf':
                text_content = extract_text_from_pdf(BytesIO(file.read()))
            elif file_ext in ['xlsx', 'xls']:
                text_content = extract_text_from_excel(BytesIO(file.read()))
            elif file_ext in ['txt', 'csv']:
                text_content = file.read().decode('utf-8', errors='ignore')

            print(f"✅ Arquivo processado: {len(text_content)} caracteres")

            return jsonify({
                'success': True,
                'filename': filename,
                'content': text_content,
                'content_length': len(text_content)
            })

        return jsonify({'error': 'Tipo de arquivo não permitido'}), 400

    except Exception as e:
        print(f"❌ Erro no upload: {str(e)}")
        return jsonify({
            'success': False,
            'error': str(e),
            'message': 'Erro ao processar arquivo'
        }), 500


@app.route('/upload_marketing_data', methods=['POST'])
def upload_marketing_data():
    """Endpoint de upload específico para dados de marketing (CSV)"""
    try:
        if 'file' not in request.files:
            return jsonify({'error': 'Nenhum arquivo enviado'}), 400

        file = request.files['file']

        if file.filename == '':
            return jsonify({'error': 'Nome de arquivo vazio'}), 400

        filename = secure_filename(file.filename)

        if not filename.endswith('.csv'):
            return jsonify({'error': 'Apenas arquivos CSV são aceitos para dados de marketing'}), 400

        print(f"📊 Carregando dados de marketing: {filename}")

        # Ler CSV
        df = pd.read_csv(BytesIO(file.read()))

        print(f"✅ Dataset carregado: {len(df)} linhas, {len(df.columns)} colunas")
        print(f"📋 Colunas: {list(df.columns)}")

        # Processar dados com vetorização
        processed_df, cluster_analysis = process_marketing_data(df)

        # Salvar dataset processado
        output_path = os.path.join(app.config['DATA_FOLDER'], 'marketing_data_processed.csv')
        processed_df.to_csv(output_path, index=False)

        return jsonify({
            'success': True,
            'filename': filename,
            'rows': len(processed_df),
            'columns': list(processed_df.columns),
            'clusters': int(processed_df['text_cluster'].nunique()),
            'cluster_analysis': cluster_analysis,
            'message': 'Dados processados e vetorizados com sucesso!'
        })

    except Exception as e:
        print(f"❌ Erro ao processar dados de marketing: {str(e)}")
        return jsonify({
            'success': False,
            'error': str(e),
            'message': 'Erro ao processar dados de marketing'
        }), 500


@app.route('/semantic_search', methods=['POST'])
def api_semantic_search():
    """Endpoint de busca semântica"""
    try:
        data = request.json
        query = data.get('query')
        top_n = data.get('top_n', 5)

        if not query:
            return jsonify({'error': 'Query não fornecida'}), 400

        if marketing_data is None:
            return jsonify({'error': 'Nenhum dataset de marketing carregado'}), 400

        print(f"🔍 Busca semântica: '{query}'")

        results = semantic_search(query, top_n)

        if results is None:
            return jsonify({'error': 'Erro na busca semântica'}), 500

        # Converter para formato JSON-friendly
        results_dict = results.to_dict('records')

        # Limpar embeddings para resposta (muito grande)
        for item in results_dict:
            if 'embedding' in item:
                del item['embedding']

        return jsonify({
            'success': True,
            'query': query,
            'results': results_dict,
            'count': len(results_dict)
        })

    except Exception as e:
        print(f"❌ Erro na busca semântica: {str(e)}")
        return jsonify({
            'success': False,
            'error': str(e)
        }), 500


@app.route('/find_similar', methods=['POST'])
def api_find_similar():
    """Endpoint para encontrar itens similares"""
    try:
        data = request.json
        index = data.get('index')
        top_n = data.get('top_n', 5)

        if index is None:
            return jsonify({'error': 'Índice não fornecido'}), 400

        if marketing_data is None:
            return jsonify({'error': 'Nenhum dataset de marketing carregado'}), 400

        print(f"🔍 Buscando similares ao índice {index}")

        results = find_similar_items(index, top_n)

        if results is None:
            return jsonify({'error': 'Erro ao buscar similares'}), 500

        results_dict = results.to_dict('records')

        # Limpar embeddings
        for item in results_dict:
            if 'embedding' in item:
                del item['embedding']

        return jsonify({
            'success': True,
            'reference_index': index,
            'results': results_dict,
            'count': len(results_dict)
        })

    except Exception as e:
        print(f"❌ Erro ao buscar similares: {str(e)}")
        return jsonify({
            'success': False,
            'error': str(e)
        }), 500


@app.route('/get_clusters', methods=['GET'])
def get_clusters():
    """Retorna informações sobre clusters"""
    try:
        if marketing_data is None:
            return jsonify({'error': 'Nenhum dataset de marketing carregado'}), 400

        cluster_analysis = marketing_data['cluster_analysis']
        df = marketing_data['dataframe']

        # Informações adicionais sobre clusters
        cluster_info = []
        for cluster_id in df['text_cluster'].unique():
            cluster_items = df[df['text_cluster'] == cluster_id]
            cluster_info.append({
                'cluster_id': int(cluster_id),
                'size': len(cluster_items),
                'sample_items': cluster_items.head(3).to_dict('records')
            })

        # Limpar embeddings das amostras
        for cluster in cluster_info:
            for item in cluster['sample_items']:
                if 'embedding' in item:
                    del item['embedding']

        return jsonify({
            'success': True,
            'total_items': len(df),
            'num_clusters': int(df['text_cluster'].nunique()),
            'cluster_statistics': cluster_analysis,
            'cluster_details': cluster_info
        })

    except Exception as e:
        print(f"❌ Erro ao obter clusters: {str(e)}")
        return jsonify({
            'success': False,
            'error': str(e)
        }), 500


@app.route('/get_analysis', methods=['GET'])
def get_analysis():
    """Retorna análise completa do dataset"""
    try:
        if marketing_data is None:
            return jsonify({'error': 'Nenhum dataset de marketing carregado'}), 400

        df = marketing_data['dataframe']

        # Estatísticas descritivas
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        stats = df[numeric_cols].describe().to_dict()

        # Correlações
        correlations = df[numeric_cols].corr().to_dict()

        return jsonify({
            'success': True,
            'total_items': len(df),
            'columns': list(df.columns),
            'statistics': stats,
            'correlations': correlations,
            'clusters': int(df['text_cluster'].nunique())
        })

    except Exception as e:
        print(f"❌ Erro ao obter análise: {str(e)}")
        return jsonify({
            'success': False,
            'error': str(e)
        }), 500


@app.route('/analyze', methods=['POST'])
def analyze():
    """Endpoint de análise com Gemini AI"""
    try:
        data = request.json
        api_key = data.get('api_key')
        context_data = data.get('context_data')

        if not api_key:
            return jsonify({'error': 'API Key não fornecida'}), 400

        if not context_data:
            return jsonify({'error': 'Dados para análise não fornecidos'}), 400

        print(f"🤖 Iniciando análise com Gemini AI...")
        print(f"📊 Dados para análise: {len(context_data)} caracteres")

        genai.configure(api_key=api_key)
        model = genai.GenerativeModel('gemini-2.0-flash-exp')

        prompt = f"""
Você é um analista de mercado sênior especializado em marketing digital e ciência de dados.
Analise os dados fornecidos e retorne um JSON estruturado com:

1. resumo: Um resumo executivo claro e objetivo (2-3 parágrafos)
2. pontosFortes: Lista de 3-5 pontos fortes identificados
3. pontosFracos: Lista de 3-5 riscos ou fraquezas identificados
4. estrategia: Uma recomendação estratégica clara e acionável
5. insights: Lista de 3-4 insights-chave adicionais
6. metricas: Objeto com métricas importantes extraídas (se disponível)
7. recomendacoes: Lista de ações específicas recomendadas

DADOS PARA ANÁLISE:
{context_data}

Retorne APENAS o JSON, sem markdown ou texto adicional.
"""

        response = model.generate_content(
            prompt,
            generation_config={
                'temperature': 0.7,
                'top_p': 0.95,
                'top_k': 40,
                'max_output_tokens': 2048,
            }
        )

        response_text = response.text.strip()

        if response_text.startswith('```json'):
            response_text = response_text.replace('```json', '').replace('```', '').strip()
        elif response_text.startswith('```'):
            response_text = response_text.replace('```', '').strip()

        analysis_result = json.loads(response_text)

        print(f"✅ Análise concluída com sucesso!")

        return jsonify({
            'success': True,
            'analysis': analysis_result
        })

    except json.JSONDecodeError as e:
        print(f"❌ Erro ao parsear resposta da IA: {str(e)}")
        return jsonify({
            'success': False,
            'error': 'Erro ao parsear resposta da IA',
            'details': str(e)
        }), 500

    except Exception as e:
        print(f"❌ Erro na análise: {str(e)}")
        return jsonify({
            'success': False,
            'error': str(e),
            'message': 'Erro ao processar análise'
        }), 500


@app.route('/health', methods=['GET'])
def health_check():
    """Endpoint de health check"""
    return jsonify({
        'status': 'healthy',
        'service': 'Market Intelligence Pro - Unified',
        'uptime': 'active',
        'embedding_model': 'all-MiniLM-L6-v2',
        'marketing_data_loaded': marketing_data is not None
    })


# ============================================
# INICIALIZAÇÃO DO SERVIDOR
# ============================================

def run_server(port=5000):
    """Inicia o servidor Flask"""
    app.run(host='0.0.0.0', port=port, debug=False, use_reloader=False)


# Iniciar servidor em background
from google.colab.output import serve_kernel_port_as_iframe

thread = threading.Thread(target=lambda: run_server(5000))
thread.daemon = True
thread.start()

print("\n" + "="*60)
print("🎉 SERVIDOR INICIADO COM SUCESSO!")
print("="*60)
print("\n📡 Endpoints disponíveis:")
print("\n  🌐 Web Scraping:")
print("     • POST /scrape - Web scraping de URLs")
print("\n  📁 Upload & Processamento:")
print("     • POST /upload - Upload de arquivos gerais (PDF, Excel, CSV)")
print("     • POST /upload_marketing_data - Upload de dados de marketing (CSV)")
print("\n  🤖 Análise com IA:")
print("     • POST /analyze - Análise com IA Gemini")
print("\n  🔍 Busca & Similaridade Semântica:")
print("     • POST /semantic_search - Busca semântica no dataset")
print("     • POST /find_similar - Encontrar itens similares")
print("\n  📊 Análise de Dados:")
print("     • GET  /get_clusters - Informações sobre clusters")
print("     • GET  /get_analysis - Análise estatística completa")
print("\n  ❤️  Sistema:")
print("     • GET  /health - Status do servidor")
print("\n🔗 URL do servidor: http://localhost:5000")
print("\n" + "="*60)
print("\n📝 GUIA DE USO RÁPIDO:")
print("\n1️⃣  CONFIGURAÇÃO INICIAL:")
print("   • Obtenha sua Gemini API Key: https://aistudio.google.com/app/apikey")
print("\n2️⃣  UPLOAD DE DADOS DE MARKETING:")
print("   • Prepare um arquivo CSV com suas campanhas de marketing")
print("   • Colunas sugeridas: ad_copy, campaign_name, impressions, clicks, conversions, spend")
print("   • Faça upload via POST /upload_marketing_data")
print("\n3️⃣  ANÁLISE SEMÂNTICA:")
print("   • Use /semantic_search para buscar campanhas similares")
print("   • Use /find_similar para encontrar anúncios parecidos")
print("   • Use /get_clusters para ver agrupamentos automáticos")
print("\n4️⃣  EXEMPLO DE USO EM PYTHON:")
print("""
   import requests

   # Upload de dados
   with open('marketing_data.csv', 'rb') as f:
       response = requests.post('http://localhost:5000/upload_marketing_data',
                               files={'file': f})
   print(response.json())

   # Busca semântica
   response = requests.post('http://localhost:5000/semantic_search',
                           json={'query': 'promoção desconto', 'top_n': 5})
   print(response.json())

   # Análise com IA
   response = requests.post('http://localhost:5000/analyze',
                           json={'api_key': 'SUA_KEY',
                                 'context_data': 'dados para análise...'})
   print(response.json())
""")
print("\n" + "="*60)
print("\n✨ RECURSOS DISPONÍVEIS:")
print("   ✓ Web Scraping com Playwright")
print("   ✓ Processamento de PDF, Excel e CSV")
print("   ✓ Análise com IA Gemini 2.0")
print("   ✓ Vetorização semântica (Sentence Transformers)")
print("   ✓ Clusterização automática de campanhas")
print("   ✓ Busca por similaridade")
print("   ✓ Cálculo automático de métricas (CTR, CPA, CPC)")
print("   ✓ Análise estatística e visualizações")
print("\n" + "="*60)

# Exibir servidor no iframe do Colab
serve_kernel_port_as_iframe(5000)


🚀 Instalando dependências do Market Intelligence Pro Unificado...
Installing dependencies...
Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
f

<IPython.core.display.Javascript object>

 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.


In [4]:
"""
Script de Teste e Demonstração
Market Intelligence Pro - Versão Unificada

Este script demonstra todas as funcionalidades do sistema.
Execute após iniciar o servidor no Google Colab.
"""

import requests
import json
import time

# ============================================
# CONFIGURAÇÕES
# ============================================

BASE_URL = 'http://localhost:5000'
GEMINI_API_KEY = 'SUA_GEMINI_API_KEY_AQUI'  # ⚠️ SUBSTITUA pela sua key

# ============================================
# FUNÇÕES AUXILIARES
# ============================================

def print_section(title):
    """Imprime cabeçalho de seção"""
    print("\n" + "="*60)
    print(f"  {title}")
    print("="*60 + "\n")


def print_json(data, indent=2):
    """Imprime JSON formatado"""
    print(json.dumps(data, indent=indent, ensure_ascii=False))


# ============================================
# TESTES
# ============================================

def test_health_check():
    """Teste 1: Health Check"""
    print_section("TEST 1: Health Check")

    response = requests.get(f'{BASE_URL}/health')
    data = response.json()

    print(f"Status: {data['status']}")
    print(f"Serviço: {data['service']}")
    print(f"Modelo de embeddings: {data['embedding_model']}")
    print(f"Dados carregados: {data['marketing_data_loaded']}")

    assert response.status_code == 200
    assert data['status'] == 'healthy'
    print("\n✅ Health check passou!")


def test_web_scraping():
    """Teste 2: Web Scraping"""
    print_section("TEST 2: Web Scraping")

    test_url = 'https://example.com'
    print(f"🌐 Fazendo scraping de: {test_url}")

    response = requests.post(f'{BASE_URL}/scrape',
                            json={'url': test_url})
    data = response.json()

    if data['success']:
        print(f"✅ Scraping concluído!")
        print(f"Título: {data['title']}")
        print(f"Conteúdo: {data['content'][:200]}...")
        print(f"Total de caracteres: {data['metadata']['content_length']}")
    else:
        print(f"❌ Erro: {data.get('error', 'Desconhecido')}")


def test_upload_marketing_data():
    """Teste 3: Upload de dados de marketing"""
    print_section("TEST 3: Upload de Dados de Marketing")

    csv_path = 'marketing_data_example.csv'

    try:
        with open(csv_path, 'rb') as f:
            files = {'file': f}
            response = requests.post(f'{BASE_URL}/upload_marketing_data',
                                   files=files)

        data = response.json()

        if data['success']:
            print(f"✅ Dataset carregado com sucesso!")
            print(f"Arquivo: {data['filename']}")
            print(f"Linhas: {data['rows']}")
            print(f"Colunas: {', '.join(data['columns'])}")
            print(f"Clusters formados: {data['clusters']}")

            print("\n📊 Análise por cluster:")
            for cluster in data['cluster_analysis']:
                print(f"\nCluster {cluster['text_cluster']}:")
                print(f"  Impressões médias: {cluster.get('impressions', 0):,.0f}")
                print(f"  Clicks médios: {cluster.get('clicks', 0):,.0f}")
                print(f"  CTR médio: {cluster.get('ctr', 0):.2%}")
        else:
            print(f"❌ Erro: {data.get('error', 'Desconhecido')}")

    except FileNotFoundError:
        print(f"⚠️  Arquivo {csv_path} não encontrado.")
        print("   Certifique-se de que o arquivo está no mesmo diretório.")


def test_semantic_search():
    """Teste 4: Busca Semântica"""
    print_section("TEST 4: Busca Semântica")

    queries = [
        "curso online desconto promoção",
        "programação Python iniciantes",
        "marketing digital redes sociais"
    ]

    for query in queries:
        print(f"\n🔍 Query: '{query}'")

        response = requests.post(f'{BASE_URL}/semantic_search',
                               json={'query': query, 'top_n': 3})

        if response.status_code == 200:
            data = response.json()

            if data['success']:
                print(f"✅ Encontrados {data['count']} resultados:\n")

                for i, result in enumerate(data['results'], 1):
                    print(f"{i}. {result.get('ad_copy', 'N/A')[:60]}...")
                    print(f"   Campanha: {result.get('campaign_name', 'N/A')}")
                    print(f"   Similaridade: {result.get('similarity_score', 0):.2%}")
                    print(f"   CTR: {result.get('ctr', 0):.2%}")
                    print()
            else:
                print(f"❌ Erro: {data.get('error', 'Desconhecido')}")
        else:
            print(f"⚠️  Nenhum dataset carregado. Execute test_upload_marketing_data() primeiro.")
            break


def test_find_similar():
    """Teste 5: Encontrar Similares"""
    print_section("TEST 5: Encontrar Anúncios Similares")

    test_index = 0
    print(f"🔗 Buscando similares ao índice {test_index}...")

    response = requests.post(f'{BASE_URL}/find_similar',
                           json={'index': test_index, 'top_n': 5})

    if response.status_code == 200:
        data = response.json()

        if data['success']:
            print(f"✅ Encontrados {data['count']} similares:\n")

            for i, result in enumerate(data['results'], 1):
                print(f"{i}. {result.get('ad_copy', 'N/A')[:60]}...")
                print(f"   Campanha: {result.get('campaign_name', 'N/A')}")
                print(f"   CTR: {result.get('ctr', 0):.2%}")
                print()
        else:
            print(f"❌ Erro: {data.get('error', 'Desconhecido')}")
    else:
        print(f"⚠️  Erro na requisição.")


def test_get_clusters():
    """Teste 6: Análise de Clusters"""
    print_section("TEST 6: Análise de Clusters")

    response = requests.get(f'{BASE_URL}/get_clusters')

    if response.status_code == 200:
        data = response.json()

        if data['success']:
            print(f"✅ Análise de clusters:")
            print(f"Total de itens: {data['total_items']}")
            print(f"Número de clusters: {data['num_clusters']}\n")

            print("📊 Detalhes por cluster:")
            for cluster in data['cluster_details']:
                print(f"\nCluster {cluster['cluster_id']}:")
                print(f"  Tamanho: {cluster['size']} itens")
                print(f"  Amostras:")

                for item in cluster['sample_items'][:2]:
                    print(f"    • {item.get('ad_copy', 'N/A')[:50]}...")
        else:
            print(f"❌ Erro: {data.get('error', 'Desconhecido')}")
    else:
        print(f"⚠️  Erro na requisição.")


def test_get_analysis():
    """Teste 7: Análise Estatística"""
    print_section("TEST 7: Análise Estatística Completa")

    response = requests.get(f'{BASE_URL}/get_analysis')

    if response.status_code == 200:
        data = response.json()

        if data['success']:
            print(f"✅ Análise estatística:")
            print(f"Total de itens: {data['total_items']}")
            print(f"Clusters: {data['clusters']}")
            print(f"\n📊 Estatísticas das métricas principais:\n")

            metrics = ['impressions', 'clicks', 'conversions', 'ctr']

            for metric in metrics:
                if metric in data['statistics']:
                    stats = data['statistics'][metric]
                    print(f"{metric.upper()}:")
                    print(f"  Média: {stats.get('mean', 0):,.2f}")
                    print(f"  Desvio padrão: {stats.get('std', 0):,.2f}")
                    print(f"  Min: {stats.get('min', 0):,.2f}")
                    print(f"  Max: {stats.get('max', 0):,.2f}")
                    print()
        else:
            print(f"❌ Erro: {data.get('error', 'Desconhecido')}")
    else:
        print(f"⚠️  Erro na requisição.")


def test_ai_analysis():
    """Teste 8: Análise com IA Gemini"""
    print_section("TEST 8: Análise com IA Gemini")

    if GEMINI_API_KEY == 'SUA_GEMINI_API_KEY_AQUI':
        print("⚠️  Configure sua GEMINI_API_KEY no início do script.")
        print("   Obtenha em: https://aistudio.google.com/app/apikey")
        return

    # Obter dados de um cluster para análise
    clusters_response = requests.get(f'{BASE_URL}/get_clusters')

    if clusters_response.status_code != 200:
        print("⚠️  Carregue os dados primeiro.")
        return

    clusters_data = clusters_response.json()

    if not clusters_data['success']:
        print("⚠️  Erro ao obter clusters.")
        return

    # Preparar contexto
    cluster_0 = clusters_data['cluster_details'][0]
    context = f"""
Análise de Cluster de Campanhas de Marketing:

Cluster ID: {cluster_0['cluster_id']}
Tamanho: {cluster_0['size']} campanhas

Exemplos de anúncios neste cluster:
"""

    for item in cluster_0['sample_items'][:3]:
        context += f"\n- {item.get('ad_copy', 'N/A')}"
        context += f"\n  Impressões: {item.get('impressions', 0):,}"
        context += f"\n  Clicks: {item.get('clicks', 0):,}"
        context += f"\n  CTR: {item.get('ctr', 0):.2%}\n"

    print("🤖 Enviando para análise com Gemini AI...")
    print(f"Contexto: {len(context)} caracteres\n")

    response = requests.post(f'{BASE_URL}/analyze',
                           json={
                               'api_key': GEMINI_API_KEY,
                               'context_data': context
                           })

    if response.status_code == 200:
        data = response.json()

        if data['success']:
            analysis = data['analysis']

            print("✅ Análise concluída!\n")

            print("📝 RESUMO EXECUTIVO:")
            print(analysis.get('resumo', 'N/A'))

            print("\n💪 PONTOS FORTES:")
            for ponto in analysis.get('pontosFortes', []):
                print(f"  • {ponto}")

            print("\n⚠️  PONTOS FRACOS:")
            for ponto in analysis.get('pontosFracos', []):
                print(f"  • {ponto}")

            print("\n🎯 ESTRATÉGIA RECOMENDADA:")
            print(analysis.get('estrategia', 'N/A'))

            print("\n💡 INSIGHTS:")
            for insight in analysis.get('insights', []):
                print(f"  • {insight}")

            if 'recomendacoes' in analysis:
                print("\n📋 RECOMENDAÇÕES:")
                for rec in analysis['recomendacoes']:
                    print(f"  • {rec}")
        else:
            print(f"❌ Erro: {data.get('error', 'Desconhecido')}")
    else:
        print(f"⚠️  Erro na requisição: {response.status_code}")


# ============================================
# EXECUÇÃO DOS TESTES
# ============================================

def run_all_tests():
    """Executa todos os testes"""
    print("\n" + "🚀 "  + "="*58)
    print("  MARKET INTELLIGENCE PRO - UNIFIED")
    print("  Suite de Testes Completa")
    print("🚀 " + "="*58 + "\n")

    time.sleep(1)

    tests = [
        ("Health Check", test_health_check),
        ("Web Scraping", test_web_scraping),
        ("Upload de Dados de Marketing", test_upload_marketing_data),
        ("Busca Semântica", test_semantic_search),
        ("Encontrar Similares", test_find_similar),
        ("Análise de Clusters", test_get_clusters),
        ("Análise Estatística", test_get_analysis),
        ("Análise com IA", test_ai_analysis)
    ]

    passed = 0
    failed = 0

    for name, test_func in tests:
        try:
            test_func()
            passed += 1
            time.sleep(1)  # Pausa entre testes
        except Exception as e:
            print(f"\n❌ Teste '{name}' falhou: {str(e)}")
            failed += 1
            time.sleep(1)

    # Resumo
    print("\n" + "="*60)
    print("  RESUMO DOS TESTES")
    print("="*60)
    print(f"\n✅ Passaram: {passed}")
    print(f"❌ Falharam: {failed}")
    print(f"📊 Total: {len(tests)}")
    print(f"\nTaxa de sucesso: {(passed/len(tests)*100):.1f}%")
    print("\n" + "="*60 + "\n")


# ============================================
# MENU INTERATIVO
# ============================================

def show_menu():
    """Exibe menu interativo"""
    print("\n" + "="*60)
    print("  MENU DE TESTES")
    print("="*60)
    print("\n1. Executar todos os testes")
    print("2. Health Check")
    print("3. Web Scraping")
    print("4. Upload de Dados de Marketing")
    print("5. Busca Semântica")
    print("6. Encontrar Similares")
    print("7. Análise de Clusters")
    print("8. Análise Estatística")
    print("9. Análise com IA Gemini")
    print("0. Sair")
    print("\n" + "="*60)


def main():
    """Função principal"""
    while True:
        show_menu()
        choice = input("\nEscolha uma opção: ")

        if choice == '1':
            run_all_tests()
        elif choice == '2':
            test_health_check()
        elif choice == '3':
            test_web_scraping()
        elif choice == '4':
            test_upload_marketing_data()
        elif choice == '5':
            test_semantic_search()
        elif choice == '6':
            test_find_similar()
        elif choice == '7':
            test_get_clusters()
        elif choice == '8':
            test_get_analysis()
        elif choice == '9':
            test_ai_analysis()
        elif choice == '0':
            print("\n👋 Até logo!")
            break
        else:
            print("\n⚠️  Opção inválida!")

        input("\nPressione Enter para continuar...")


if __name__ == '__main__':
    print("""
╔════════════════════════════════════════════════════════════╗
║                                                            ║
║     MARKET INTELLIGENCE PRO - UNIFIED                      ║
║     Script de Teste e Demonstração                        ║
║                                                            ║
║     Certifique-se de que o servidor está rodando em:      ║
║     http://localhost:5000                                  ║
║                                                            ║
╚════════════════════════════════════════════════════════════╝
    """)

    # Verificar se servidor está online
    try:
        response = requests.get(f'{BASE_URL}/health', timeout=2)
        if response.status_code == 200:
            print("✅ Servidor online e pronto!\n")
            main()
        else:
            print("⚠️  Servidor não respondeu corretamente.")
    except requests.exceptions.RequestException:
        print("❌ Erro: Servidor não está rodando!")
        print("   Inicie o servidor no Google Colab primeiro.")


INFO:werkzeug:127.0.0.1 - - [16/Jan/2026 21:21:45] "GET /health HTTP/1.1" 200 -



╔════════════════════════════════════════════════════════════╗
║                                                            ║
║     MARKET INTELLIGENCE PRO - UNIFIED                      ║
║     Script de Teste e Demonstração                        ║
║                                                            ║
║     Certifique-se de que o servidor está rodando em:      ║
║     http://localhost:5000                                  ║
║                                                            ║
╚════════════════════════════════════════════════════════════╝
    
✅ Servidor online e pronto!


  MENU DE TESTES

1. Executar todos os testes
2. Health Check
3. Web Scraping
4. Upload de Dados de Marketing
5. Busca Semântica
6. Encontrar Similares
7. Análise de Clusters
8. Análise Estatística
9. Análise com IA Gemini
0. Sair


Escolha uma opção: 2


INFO:werkzeug:127.0.0.1 - - [16/Jan/2026 21:21:55] "GET /health HTTP/1.1" 200 -



  TEST 1: Health Check

Status: healthy
Serviço: Market Intelligence Pro - Unified
Modelo de embeddings: all-MiniLM-L6-v2
Dados carregados: False

✅ Health check passou!

Pressione Enter para continuar...

  MENU DE TESTES

1. Executar todos os testes
2. Health Check
3. Web Scraping
4. Upload de Dados de Marketing
5. Busca Semântica
6. Encontrar Similares
7. Análise de Clusters
8. Análise Estatística
9. Análise com IA Gemini
0. Sair


Escolha uma opção: 4

  TEST 3: Upload de Dados de Marketing

⚠️  Arquivo marketing_data_example.csv não encontrado.
   Certifique-se de que o arquivo está no mesmo diretório.


KeyboardInterrupt: Interrupted by user